# Etapa 5 - Redes Bayesianas

Nesta etapa montamos uma Rede Bayesiana com as variáveis do domínio de Meio Ambiente (autos de infração do IBAMA, 2021). Fazemos o grafo (DAG), estimamos as tabelas de probabilidade condicional (CPTs) a partir dos dados, fazemos inferência (por enumeração feita do zero e com o pgmpy), verificamos duas independências condicionais com d-separação e calculamos a economia de parâmetros.

Antes de executar: o notebook precisa do arquivo tabela_p4.csv (feito na Etapa 1). Ele procura o arquivo na pasta atual, em dados/ e em ../dados/. Se não achar, baixa do GitHub do grupo. O notebook cria o arquivo probs_rede.csv, que é usado na Etapa 6.

Bibliotecas: pgmpy, pandas e numpy.

In [6]:
%pip install pgmpy

In [7]:
import os
import itertools
import urllib.request
import numpy as np
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

np.random.seed(42)

URL_BASE = "https://raw.githubusercontent.com/rafaellydosanjos/2o-Trab-SimbML-Incerteza_equipe_3/main/dados/"


def obter(nome):
    """Usa o arquivo local se existir. Se não, baixa do GitHub."""
    for pasta in ['.', 'dados', '../dados']:
        caminho = os.path.join(pasta, nome)
        if os.path.exists(caminho):
            return caminho
    try:
        os.makedirs('dados', exist_ok=True)
        caminho = os.path.join('dados', nome)
        urllib.request.urlretrieve(URL_BASE + nome, caminho)
        print('Baixado do GitHub:', nome)
        return caminho
    except Exception as erro:
        print('Não consegui baixar', nome, '->', erro)
        return None


CAMINHO_TABELA = obter('tabela_p4.csv')
print('tabela_p4.csv ->', CAMINHO_TABELA)
if CAMINHO_TABELA is None:
    raise FileNotFoundError('Falta o arquivo tabela_p4.csv. Veja o texto no início do notebook.')

df_full = pd.read_csv(CAMINHO_TABELA, encoding='utf-8-sig')
df_original = df_full.copy()
variaveis_rede = ['CHUVA_FAIXA', 'TEMP_FAIXA', 'BIOMA', 'TIPO_ACAO', 'Multa_Alta']
df_treino = df_full[variaveis_rede].dropna(subset=variaveis_rede).astype(str)
print('Linhas usadas:', len(df_treino))

tabela_p4.csv -> dados/tabela_p4.csv
Linhas usadas: 2618


## Passo 1 - O grafo (DAG)

Escolhemos 5 variáveis:

1. CHUVA_FAIXA: faixa de chuva do mês e do estado (INMET).
2. TEMP_FAIXA: faixa de temperatura do mês e do estado (INMET).
3. BIOMA: bioma onde ocorreu a infração.
4. TIPO_ACAO: tipo de ação do IBAMA (Rotina, Operação ou Outros).
5. Multa_Alta: variável alvo (1 = multa acima da mediana, 0 = abaixo).

Arestas e justificativas (são hipóteses nossas, não provas):

- CHUVA_FAIXA -> TIPO_ACAO e TEMP_FAIXA -> TIPO_ACAO: o clima pode influenciar quando a equipe consegue ir a campo, e por isso o tipo de ação.
- BIOMA -> TIPO_ACAO: o tipo de ação pode mudar de uma região para outra.
- TIPO_ACAO -> Multa_Alta: uma operação planejada pode encontrar infrações maiores que uma ação de rotina.
- BIOMA -> Multa_Alta: o bioma tem relação forte com o valor da multa (na Amazônia, 67,6% das multas são altas). Sem esta aresta, o risco da rede tinha só 3 valores diferentes.

Limitação: como Multa_Alta só tem BIOMA e TIPO_ACAO como pais, a chuva e a temperatura não mudam a probabilidade de multa alta que sai da rede.

In [8]:
modelo = DiscreteBayesianNetwork([
    ('CHUVA_FAIXA', 'TIPO_ACAO'),
    ('TEMP_FAIXA', 'TIPO_ACAO'),
    ('BIOMA', 'TIPO_ACAO'),
    ('TIPO_ACAO', 'Multa_Alta'),
    ('BIOMA', 'Multa_Alta'),
])
print('Arestas:', len(modelo.edges()))
print(sorted(modelo.edges()))

Arestas: 5
[('BIOMA', 'Multa_Alta'), ('BIOMA', 'TIPO_ACAO'), ('CHUVA_FAIXA', 'TIPO_ACAO'), ('TEMP_FAIXA', 'TIPO_ACAO'), ('TIPO_ACAO', 'Multa_Alta')]


## Passo 2 - Estimar as CPTs a partir dos dados

Usamos o BayesianEstimator do pgmpy com a suavização BDeu (equivalent_sample_size = 10). Ela soma uma contagem pequena em cada combinação, para que nenhuma probabilidade fique igual a 0 quando uma combinação não aparece nos dados.

In [9]:
estimator = BayesianEstimator(modelo, df_treino)
cpts = estimator.get_parameters(prior_type='BDeu', equivalent_sample_size=10)

for cpd in cpts:
    modelo.add_cpds(cpd)

print('Modelo é válido?', modelo.check_model())
print()
print('CPT de BIOMA:')
print(modelo.get_cpds('BIOMA'))

Modelo é válido? True

CPT de BIOMA:


/tmp/ipykernel_1153/2714226092.py:1: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  estimator = BayesianEstimator(modelo, df_treino)


+---------------------------+------------+
| BIOMA(Amazonia)           | 0.609372   |
+---------------------------+------------+
| BIOMA(Caatinga)           | 0.106327   |
+---------------------------+------------+
| BIOMA(Cerrado)            | 0.105947   |
+---------------------------+------------+
| BIOMA(Costeiro e Marinho) | 0.0028267  |
+---------------------------+------------+
| BIOMA(Mata Atlantica)     | 0.156556   |
+---------------------------+------------+
| BIOMA(Pampa)              | 0.00472929 |
+---------------------------+------------+
| BIOMA(Pantanal)           | 0.0142422  |
+---------------------------+------------+


## Passo 3 - Inferência com o pgmpy

Consulta: qual a probabilidade de a multa ser alta sabendo que a ação foi uma Operação? Usamos a Eliminação de Variáveis do pgmpy (VariableElimination).

In [10]:
inferencia = VariableElimination(modelo)

resultado = inferencia.query(variables=['Multa_Alta'], evidence={'TIPO_ACAO': 'Operação'}, show_progress=False)
print('P(Multa_Alta | TIPO_ACAO = Operação), pgmpy:')
print(resultado)

P(Multa_Alta | TIPO_ACAO = Operação), pgmpy:
+---------------+-------------------+
| Multa_Alta    |   phi(Multa_Alta) |
+===============+===================+
| Multa_Alta(0) |            0.4942 |
+---------------+-------------------+
| Multa_Alta(1) |            0.5058 |
+---------------+-------------------+


## Passo 4 - Inferência por enumeração, feita do zero

Na enumeração, a probabilidade conjunta é o produto das CPTs: P(x1, ..., xn) = produto de P(xi | pais de xi). Para responder uma consulta, somamos essa conjunta sobre todas as variáveis que não estão na consulta nem na evidência, e depois normalizamos para a soma dar 1.

A função abaixo usa as CPTs que o pgmpy estimou, mas faz a soma e a normalização sem usar o motor de inferência dele. Depois comparamos com o resultado do pgmpy.

In [11]:
def prob_cpd(modelo, var, estado, atrib):
    """Lê da CPT o valor de P(var = estado | pais), com os pais dados em atrib."""
    cpd = modelo.get_cpds(var)
    idx = [cpd.state_names[var].index(estado)]
    for pai in cpd.variables[1:]:
        idx.append(cpd.state_names[pai].index(atrib[pai]))
    return float(cpd.values[tuple(idx)])


def enumeracao(modelo, consulta, evidencia):
    """Devolve P(consulta | evidencia) por enumeração."""
    estados = {v: modelo.get_cpds(v).state_names[v] for v in modelo.nodes()}
    ocultas = [v for v in modelo.nodes() if v != consulta and v not in evidencia]
    resultado = {}
    for q in estados[consulta]:
        total = 0.0
        for combo in itertools.product(*[estados[v] for v in ocultas]):
            atrib = dict(evidencia)
            atrib[consulta] = q
            atrib.update(zip(ocultas, combo))
            p = 1.0
            for v in modelo.nodes():
                p *= prob_cpd(modelo, v, atrib[v], atrib)
            total += p
        resultado[q] = total
    z = sum(resultado.values())
    return {q: p / z for q, p in resultado.items()}


def valor_pgmpy(consulta, evidencia):
    r = inferencia.query(variables=[consulta], evidence=evidencia, show_progress=False)
    return dict(zip(r.state_names[consulta], [float(x) for x in r.values]))


consultas = [
    {'TIPO_ACAO': 'Operação'},
    {'BIOMA': 'Amazonia', 'TIPO_ACAO': 'Rotina'},
    {'BIOMA': 'Caatinga'},
]

for evid in consultas:
    meu = enumeracao(modelo, 'Multa_Alta', evid)
    pg = valor_pgmpy('Multa_Alta', evid)
    iguais = all(np.isclose(meu[k], pg[k]) for k in meu)
    print('Evidência:', evid)
    print('  enumeração:', {k: round(v, 4) for k, v in meu.items()})
    print('  pgmpy:     ', {k: round(v, 4) for k, v in pg.items()})
    print('  iguais?', iguais)

Evidência: {'TIPO_ACAO': 'Operação'}
  enumeração: {'0': 0.4942, '1': 0.5058}
  pgmpy:      {'0': 0.4942, '1': 0.5058}
  iguais? True
Evidência: {'BIOMA': 'Amazonia', 'TIPO_ACAO': 'Rotina'}
  enumeração: {'0': 0.3582, '1': 0.6418}
  pgmpy:      {'0': 0.3582, '1': 0.6418}
  iguais? True
Evidência: {'BIOMA': 'Caatinga'}
  enumeração: {'0': 0.9244, '1': 0.0756}
  pgmpy:      {'0': 0.9244, '1': 0.0756}
  iguais? True


## Passo 5 - Independência condicional (d-separação)

Duas independências para verificar:

1. Colisor: CHUVA_FAIXA e BIOMA são pais de TIPO_ACAO. Sem observar nada, o caminho entre eles está fechado e eles são independentes. Ao observar TIPO_ACAO (o colisor), o caminho abre e eles passam a ser dependentes.
2. Cadeia: TEMP_FAIXA e Multa_Alta estão ligadas por TEMP_FAIXA -> TIPO_ACAO -> Multa_Alta, então são dependentes. Se TIPO_ACAO e BIOMA são conhecidos, todos os caminhos fecham e elas ficam independentes.

is_dconnected devolve True quando existe caminho aberto (dependentes) e False quando está fechado (independentes).

In [12]:
print('--- Colisor ---')
print('CHUVA e BIOMA, sem evidência:', modelo.is_dconnected('CHUVA_FAIXA', 'BIOMA'))
print('CHUVA e BIOMA, observando TIPO_ACAO:', modelo.is_dconnected('CHUVA_FAIXA', 'BIOMA', observed=['TIPO_ACAO']))

print()
print('--- Cadeia ---')
print('TEMP e Multa, sem evidência:', modelo.is_dconnected('TEMP_FAIXA', 'Multa_Alta'))
print('TEMP e Multa, observando TIPO_ACAO e BIOMA:', modelo.is_dconnected('TEMP_FAIXA', 'Multa_Alta', observed=['TIPO_ACAO', 'BIOMA']))

--- Colisor ---
CHUVA e BIOMA, sem evidência: False
CHUVA e BIOMA, observando TIPO_ACAO: True

--- Cadeia ---
TEMP e Multa, sem evidência: True
TEMP e Multa, observando TIPO_ACAO e BIOMA: False


## Passo 6 - Economia de parâmetros

A distribuição conjunta completa precisa de (produto das cardinalidades) - 1 números. A rede precisa só de um número livre a menos que o número de estados para cada coluna de cada CPT.

In [13]:
categorias = {var: modelo.get_cardinality(var) for var in modelo.nodes()}
print('Estados por variável:', categorias)

total_conjunta = int(np.prod(list(categorias.values()))) - 1
total_rede = sum(
    len(cpt.values.flatten()) - (len(cpt.values.flatten()) // cpt.variable_card)
    for cpt in modelo.get_cpds()
)

print('Parâmetros da conjunta completa:', total_conjunta)
print('Parâmetros da Rede Bayesiana:', total_rede)
print('Economia:', total_conjunta - total_rede)

Estados por variável: {'CHUVA_FAIXA': np.int64(4), 'TIPO_ACAO': np.int64(3), 'TEMP_FAIXA': np.int64(4), 'BIOMA': np.int64(7), 'Multa_Alta': np.int64(2)}
Parâmetros da conjunta completa: 671
Parâmetros da Rede Bayesiana: 257
Economia: 414


## Passo 7 - Exportar as probabilidades para a Etapa 6

Para cada um dos 2639 autos, calculamos P(Multa_Alta = 1) dada a evidência do próprio auto (chuva, temperatura, bioma e tipo de ação). Salvamos em probs_rede.csv, na mesma pasta da tabela.

In [ ]:
p_multa_alta_lista = []

for _, row in df_original.iterrows():
    evidencia = {}
    for var in ['CHUVA_FAIXA', 'TEMP_FAIXA', 'BIOMA', 'TIPO_ACAO']:
        if pd.notna(row[var]):
            evidencia[var] = str(row[var])
    try:
        res = inferencia.query(
            variables=['Multa_Alta'],
            evidence=evidencia if len(evidencia) > 0 else None,
            show_progress=False,
        )
        idx_1 = list(res.state_names['Multa_Alta']).index('1')
        p_val = float(res.values[idx_1])
    except Exception:
        p_val = np.nan
    p_multa_alta_lista.append(p_val)

df_probs = pd.DataFrame({'p_multa_alta': p_multa_alta_lista})
print('Linhas:', len(df_probs), '| vazios (erro na consulta):', int(df_probs['p_multa_alta'].isna().sum()))

pasta_saida = os.path.dirname(CAMINHO_TABELA)
caminho_saida = os.path.join(pasta_saida, 'probs_rede.csv')
df_probs.to_csv(caminho_saida, index=False, encoding='utf-8-sig')
print('Arquivo salvo em:', caminho_saida)

novo = df_probs['p_multa_alta']
print('Mínimo', round(novo.min(), 3), '| máximo', round(novo.max(), 3), '| média', round(novo.mean(), 3))
print('Valores distintos:', novo.nunique())